# Barlow Twins — Self-Supervised Learning via Redundancy Reduction (2021)

_Papers · Self-supervised & Representation_

**Make two views' embeddings match by pushing their cross-correlation matrix toward the identity — invariance on the diagonal, decorrelation off it. No negatives.**

---

This notebook is a practice scaffold for the **Barlow Twins — Self-Supervised Learning via Redundancy Reduction (2021)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the MNIST dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.MNIST(root="./data", train=True, download=True)
print("dataset: MNIST   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Sanity-check the loss on a hand-built example

Before touching real images, we verify the Barlow Twins loss on a tiny worked case we can read by eye. The loss compares two views' embeddings through their **cross-correlation matrix** `C`: we want `1`s on the diagonal (each dimension agrees across the two views — *invariance*) and `0`s off the diagonal (different dimensions are decorrelated — *redundancy reduction*). The invariance term penalizes `(1 - C_ii)`, and the off-diagonal term, weighted by `lambda`, penalizes every `C_ij` with `i != j`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- 0. Sanity-check the worked example: cross-correlation matrix + Barlow Twins loss. ---
# N=4 images, D=2 dims, columns already mean-0 with sum-of-squares 4 (length 2). lambda = 0.5.
ZA = torch.tensor([[ 1.,  1.], [ 1., -1.], [-1.,  1.], [-1., -1.]])
ZB = torch.tensor([[ 1., -1.], [ 1.,  1.], [-1., -1.], [-1.,  1.]])

def cross_corr(zA, zB):                      # Eqn. 2: normalize each column, then z_A^T z_B
    a = (zA - zA.mean(0)) / zA.std(0, unbiased=False)
    b = (zB - zB.mean(0)) / zB.std(0, unbiased=False)
    return (a.t() @ b) / zA.shape[0]         # D x D, entries in [-1, 1]

def barlow_loss(zA, zB, lam=0.5):            # Eqn. 1
    C = cross_corr(zA, zB)
    on  = ((1 - torch.diagonal(C))**2).sum()                         # invariance term
    off = (C**2).sum() - (torch.diagonal(C)**2).sum()                # sum over i != j
    return on + lam * off, C

loss0, C0 = barlow_loss(ZA, ZB, lam=0.5)
print("worked example:  C =\n", C0.numpy(), "\n loss =", round(loss0.item(), 4))
# C = [[1. 0.] [0. -1.]] ;  loss = 4.0  (all of it from C_22 = -1: views disagree on dim 2)

### Step 2 — Define the twin encoder, projector, and the batched loss

Now we build the real network from `nn` primitives. The **encoder** is a small conv net that maps an image to a representation `h`; the **projector** is an MLP that expands `h` to a higher-dimensional embedding `z` (Barlow Twins likes large `D`). The batched `bt_loss` is the same formula as Step 1, but it normalizes each dimension across the batch (a batch-norm-style standardization) before forming the cross-correlation matrix.

In [ ]:
# --- 1. Twin encoder f and projector (built by hand from nn primitives). ---
class Encoder(nn.Module):                    # small conv net -> representation h
    def __init__(self, feat=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 14x14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 7x7
            nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.fc = nn.Linear(32, feat)
    def forward(self, x):
        return F.relu(self.fc(self.net(x)))            # h

class Projector(nn.Module):                  # MLP -> D-dim embedding z (BT likes large D)
    def __init__(self, fin=64, hid=128, out=128):
        super().__init__()
        self.l1 = nn.Linear(fin, hid)
        self.l2 = nn.Linear(hid, out)
    def forward(self, h):
        return self.l2(F.relu(self.l1(h)))            # z

# --- 2. The Barlow Twins loss on batched embeddings zA, zB (each N x D). ---
def bt_loss(zA, zB, lam=5e-3):
    eps = 1e-5
    a = (zA - zA.mean(0)) / (zA.std(0, unbiased=False) + eps)   # batch-norm each dim
    b = (zB - zB.mean(0)) / (zB.std(0, unbiased=False) + eps)
    C = (a.t() @ b) / zA.shape[0]                              # D x D cross-correlation (Eqn. 2)
    on  = ((1 - torch.diagonal(C))**2).sum()                   # diagonal -> 1  (invariance)
    off = (C**2).sum() - (torch.diagonal(C)**2).sum()          # off-diagonal -> 0 (redundancy)
    return on + lam * off                                      # Eqn. 1

### Step 3 — Pretrain on unlabelled MNIST with two augmented views

Self-supervised learning needs no labels: we make **two random augmentations** of each image and ask the embeddings of the two views to match (diagonal) while staying decorrelated (off-diagonal). We pull a 3000-image MNIST subset, keep the labels aside *only* for a later probe, and run the pretraining loop — note there are no negatives anywhere.

In [ ]:
# --- 3. Two-view augmentation + an UNLABELLED MNIST subset (torchvision, preinstalled). ---
aug  = T.Compose([T.RandomResizedCrop(28, scale=(0.5, 1.0)),
                  T.RandomApply([T.GaussianBlur(3)], 0.5), T.ToTensor()])
base = T.ToTensor()
raw  = torchvision.datasets.MNIST("./data", train=True, download=True)
idx  = np.random.RandomState(0).permutation(len(raw))[:3000]
imgs = [raw[i][0] for i in idx]
labels = torch.tensor([raw[i][1] for i in idx])     # used ONLY for the probe later

# --- 4. Pretrain Barlow Twins (no labels, no negatives). Returns the trained encoder. ---
def pretrain(lam=5e-3, epochs=15):
    enc, proj = Encoder().to(device), Projector().to(device)
    opt = torch.optim.Adam(list(enc.parameters()) + list(proj.parameters()), lr=1e-3)
    enc.train(); proj.train(); B = 256
    for ep in range(epochs):
        perm = np.random.permutation(len(imgs)); tot = 0.0; nb = 0
        for s in range(0, len(imgs), B):
            bi = perm[s:s + B]
            vA = torch.stack([aug(imgs[i]) for i in bi]).to(device)
            vB = torch.stack([aug(imgs[i]) for i in bi]).to(device)
            zA = proj(enc(vA)); zB = proj(enc(vB))
            loss = bt_loss(zA, zB, lam=lam)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item(); nb += 1
        if ep % 3 == 0: print(f"  pretrain(lam={lam}) ep {ep}  BT loss {tot/nb:.3f}")
    return enc

enc = pretrain(lam=5e-3)

### Step 4 — Evaluate: linear probe vs from-scratch, and the collapse ablation

To test the learned representation we **freeze** the encoder and train only a linear classifier on its features (the *linear-evaluation protocol*), comparing against a fresh net trained end-to-end on the same few labels. Finally we run the key ablation: setting `lambda = 0` drops the off-diagonal term, leaving only "make the views agree." Without the decorrelation pressure the embedding suffers **dimensional collapse** — feature variance plummets and the probe falls toward chance.

In [ ]:
# --- 5. FREEZE the encoder, extract features h (linear-evaluation protocol). ---
def features(encoder):
    encoder.eval()
    with torch.no_grad():
        return encoder(torch.stack([base(im) for im in imgs]).to(device)).cpu()
feats = features(enc)

def linear_probe(feats, n_lab):              # train ONLY a linear classifier on frozen h
    accs = []
    for seed in range(3):
        g = np.random.RandomState(seed)
        tr = g.permutation(len(labels))[:n_lab]; te = g.permutation(len(labels))[-600:]
        clf = nn.Linear(feats.shape[1], 10); o = torch.optim.Adam(clf.parameters(), lr=0.05)
        for _ in range(200):
            o.zero_grad(); F.cross_entropy(clf(feats[tr]), labels[tr]).backward(); o.step()
        with torch.no_grad():
            accs.append((clf(feats[te]).argmax(1) == labels[te]).float().mean().item())
    return float(np.mean(accs))

def from_scratch(n_lab):                      # train a fresh conv net end-to-end on the few labels
    accs = []
    for seed in range(3):
        torch.manual_seed(seed); g = np.random.RandomState(seed)
        tr = g.permutation(len(labels))[:n_lab]; te = g.permutation(len(labels))[-600:]
        net = nn.Sequential(Encoder(), nn.Linear(64, 10)); o = torch.optim.Adam(net.parameters(), lr=1e-3)
        Xtr = torch.stack([base(imgs[i]) for i in tr]); net.train()
        for _ in range(60):
            o.zero_grad(); F.cross_entropy(net(Xtr), labels[tr]).backward(); o.step()
        net.eval()
        with torch.no_grad():
            Xte = torch.stack([base(imgs[i]) for i in te])
            accs.append((net(Xte).argmax(1) == labels[te]).float().mean().item())
    return float(np.mean(accs))

print("\nlabels | probe(frozen BT) | from-scratch")
for n in [20, 50, 100, 300]:
    print(f"{n:6d} |      {linear_probe(feats, n):.3f}       |    {from_scratch(n):.3f}")

# --- 6. ABLATION: drop the off-diagonal (redundancy) term -> dimensional collapse. ---
enc_collapse = pretrain(lam=0.0)             # only the diagonal "make views agree" term
feats_c = features(enc_collapse)
# Collapse signature: tiny feature variance + a probe near chance (~0.10 on 10 classes).
print("\nfeature variance: BT(lam=5e-3) = %.4f   ablation(lam=0) = %.4f"
      % (feats.var(0).mean().item(), feats_c.var(0).mean().item()))
print("probe(300 labels): BT = %.3f   ablation(lam=0) = %.3f"
      % (linear_probe(feats, 300), linear_probe(feats_c, 300)))
# The frozen-BT probe beats from-scratch at every budget; the lam=0 ablation collapses
# (feature variance plummets, probe falls toward chance).
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Does a frozen Barlow Twins linear probe beat from-scratch in the low-label regime — and does removing the off-diagonal term collapse the representation?_

### Step 1 — Rebuild the self-contained pretraining pieces

This visualization cell stands on its own (it re-imports and re-defines the model) so it can be run independently. We restate the `Encoder`, `Projector`, and `bt_loss`, then load the same unlabelled MNIST subset and seed everything for reproducibility.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
np.random.seed(0)

# Pretrain a tiny Barlow Twins encoder on UNLABELLED MNIST, freeze it, then compare a linear
# probe vs from-scratch vs a lambda=0 ablation (drops the off-diagonal term -> collapse).
class Encoder(nn.Module):
    def __init__(s, feat=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                              nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                              nn.AdaptiveAvgPool2d(1), nn.Flatten())
        s.fc = nn.Linear(32, feat)
    def forward(s, x):
        return F.relu(s.fc(s.net(x)))

class Projector(nn.Module):
    def __init__(s, fin=64, hid=128, out=128):
        super().__init__()
        s.l1 = nn.Linear(fin, hid)
        s.l2 = nn.Linear(hid, out)
    def forward(s, h):
        return s.l2(F.relu(s.l1(h)))

def bt_loss(zA, zB, lam=5e-3):
    eps = 1e-5
    a = (zA - zA.mean(0)) / (zA.std(0, unbiased=False) + eps)
    b = (zB - zB.mean(0)) / (zB.std(0, unbiased=False) + eps)
    C = (a.t() @ b) / zA.shape[0]                    # D x D cross-correlation (Eqn. 2)
    on  = ((1 - torch.diagonal(C))**2).sum()         # diagonal -> 1
    off = (C**2).sum() - (torch.diagonal(C)**2).sum()  # off-diagonal -> 0
    return on + lam * off                            # Eqn. 1

aug  = T.Compose([T.RandomResizedCrop(28, scale=(0.5,1.0)),
                  T.RandomApply([T.GaussianBlur(3)], 0.5), T.ToTensor()])
base = T.ToTensor()
raw  = torchvision.datasets.MNIST("./data", train=True, download=True)
idx  = np.random.RandomState(0).permutation(len(raw))[:3000]
imgs = [raw[i][0] for i in idx]
labels = torch.tensor([raw[i][1] for i in idx])

### Step 2 — Pretrain twice and compare the probe against the collapse

We `pretrain` once with the full loss (`lambda = 5e-3`) and once with the ablation (`lambda = 0`), then measure two things: the mean feature variance (collapse shows up as near-zero variance) and the linear-probe accuracy at several tiny label budgets, against a from-scratch baseline. The expected story: the frozen Barlow Twins probe beats from-scratch everywhere, while the `lambda = 0` run collapses toward chance.

In [ ]:
def pretrain(lam, epochs=15):
    enc, proj = Encoder(), Projector()
    opt = torch.optim.Adam(list(enc.parameters())+list(proj.parameters()), lr=1e-3)
    enc.train(); proj.train(); B=256
    for ep in range(epochs):
        perm = np.random.permutation(len(imgs))
        for s0 in range(0, len(imgs), B):
            bi = perm[s0:s0+B]
            vA = torch.stack([aug(imgs[i]) for i in bi])
            vB = torch.stack([aug(imgs[i]) for i in bi])
            loss = bt_loss(proj(enc(vA)), proj(enc(vB)), lam=lam)
            opt.zero_grad(); loss.backward(); opt.step()
    enc.eval()
    with torch.no_grad():
        return enc(torch.stack([base(im) for im in imgs]))

def probe(feats, n):
    a=[]
    for seed in range(3):
        g=np.random.RandomState(seed); tr=g.permutation(len(labels))[:n]; te=g.permutation(len(labels))[-600:]
        clf=nn.Linear(feats.shape[1],10); o=torch.optim.Adam(clf.parameters(),lr=0.05)
        for _ in range(200):
            o.zero_grad(); F.cross_entropy(clf(feats[tr]),labels[tr]).backward(); o.step()
        with torch.no_grad(): a.append((clf(feats[te]).argmax(1)==labels[te]).float().mean().item())
    return round(float(np.mean(a)),3)

def scratch(n):
    a=[]
    for seed in range(3):
        torch.manual_seed(seed); g=np.random.RandomState(seed)
        tr=g.permutation(len(labels))[:n]; te=g.permutation(len(labels))[-600:]
        net=nn.Sequential(Encoder(), nn.Linear(64,10)); o=torch.optim.Adam(net.parameters(),lr=1e-3)
        Xtr=torch.stack([base(imgs[i]) for i in tr]); net.train()
        for _ in range(60):
            o.zero_grad(); F.cross_entropy(net(Xtr),labels[tr]).backward(); o.step()
        net.eval()
        with torch.no_grad():
            Xte=torch.stack([base(imgs[i]) for i in te]); a.append((net(Xte).argmax(1)==labels[te]).float().mean().item())
    return round(float(np.mean(a)),3)

feats   = pretrain(lam=5e-3)     # Barlow Twins
feats_c = pretrain(lam=0.0)      # ablation: no off-diagonal term
print("feature variance: BT=%.4f  ablation=%.4f" % (feats.var(0).mean(), feats_c.var(0).mean()))
for n in [20,50,100,300]:
    print(n, "probe(BT)", probe(feats,n), "scratch", scratch(n), "ablation(lam=0)", probe(feats_c,n))
# probe(BT) > from-scratch at every budget; lam=0 ablation collapses (variance drops, probe ~chance).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation (collapse). You pretrain Barlow Twins, freeze the encoder, and a linear probe on
            its features scores well. Then you set $\lambda = 0$ — keeping only the diagonal
            $\sum_i(1-\mathcal{C}_{ii})^2$ "make-the-views-agree" term — and retrain. The probe accuracy drops to
            near chance. What happened, and what does this prove about the off-diagonal term?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- With $\lambda=0$ the loss only rewards $\mathcal{C}_{ii}\to1$; nothing penalizes the off-diagonal. — _The model is free to make every dimension encode the same signal, since cross-dimension correlation is no longer punished._
- The $D$ dimensions correlate with each other (dimensional collapse); the embedding effectively becomes ~1-dimensional or constant. — _Redundant dimensions carry no extra information, so the representation loses almost all its capacity._
- The frozen features now separate the classes poorly, so the linear probe falls toward chance. — _A collapsed representation has nothing for the linear classifier to exploit._

**Answer:** Dropping the off-diagonal (redundancy-reduction) term removes the only force that keeps the
                 $D$ dimensions distinct, so the embedding suffers dimensional collapse — every
                 dimension encodes the same signal and the representation becomes near-useless, sending the probe
                 to chance. This proves the off-diagonal term, not the negatives (there are none), is what prevents
                 collapse in Barlow Twins. The paper's Table 5 reports the analogous ImageNet result: removing it
                 drops top-1 to ~0.1%. Our CODEVIZ panel shows the same collapse on MNIST.

</details>

**Problem 2.** Your worked example gave $\mathcal{C} = \begin{bmatrix} 1 & 0 \\ 0 & -1 \end{bmatrix}$ and loss
            $4.0$ (with $\lambda=0.5$). Now suppose training fixes dimension 2 so it agrees across views: the new
            matrix is $\mathcal{C}' = \begin{bmatrix} 1 & 0.3 \\ 0.3 & 1 \end{bmatrix}$. Recompute the loss and
            say which term now dominates.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Diagonal term: $(1-1)^2 + (1-1)^2 = 0$. — _Both diagonal entries are now $1$ — the views agree on both dimensions, so the invariance term vanishes._
- Off-diagonal term: $\mathcal{C}_{12}^2 + \mathcal{C}_{21}^2 = 0.3^2 + 0.3^2 = 0.18$. — _The two dimensions are now slightly correlated ($0.3$), so the redundancy term is nonzero._
- Total: $0 + 0.5\cdot 0.18 = 0.09$. — _Only the (weighted) off-diagonal remains; the loss dropped from $4.0$ to $0.09$._

**Answer:** The new loss is $0 + 0.5\cdot(0.3^2+0.3^2) = \mathbf{0.09}$, down from $4.0$. The invariance
                 term is now $0$ (both diagonals are $1$), so the entire (small) remaining loss comes from the
                 off-diagonal redundancy term: the two dimensions are slightly correlated ($0.3$) and the
                 loss nudges them toward independence.

</details>

**Problem 3.** A teammate argues Barlow Twins must secretly need negatives, "otherwise outputting one constant vector
            for every image makes all views match and the loss goes to zero." Where is the flaw?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check what a constant embedding does to each dimension's variance. — _A constant output has zero variance per dimension across the batch; after mean-centering each column is the zero vector._
- Check the diagonal term for a constant embedding. — _With zero-variance columns the correlation $\mathcal{C}_{ii}$ is not $1$ (it is undefined / zero), so $\sum_i(1-\mathcal{C}_{ii})^2$ stays large — the loss does NOT go to zero._
- Note the off-diagonal term independently forbids all-dimensions-equal collapse. — _Forcing $\mathcal{C}_{ij}=0$ for $i\neq j$ requires distinct, decorrelated dimensions, which a constant can never provide._

**Answer:** The flaw: a constant embedding does not drive the loss to zero. Constant outputs have
                 zero per-dimension variance, so the diagonal correlations are not $1$ and the invariance term
                 $\sum_i(1-\mathcal{C}_{ii})^2$ stays large; separately, the off-diagonal term forces the
                 dimensions to be decorrelated, which a constant cannot satisfy. Collapse is ruled out by the
                 structure of the loss, so no negatives are needed — exactly the paper's claim.

</details>